# QML as a Hybrid Job

Package a variational quantum classifier training loop as an Amazon Braket Hybrid Job: prototype the algorithm locally on `LocalSimulator`, then wrap it as an `AwsQuantumJob` with hyperparameters, CloudWatch metrics, and checkpointing.

**Objectives:**
- Build a 2-qubit variational classifier with `braket.circuits.Circuit` and estimate `<Z_0>` from measurement counts
- Train it locally with parameter-shift gradients and watch the loss fall on `LocalSimulator` (no AWS)
- Refactor the training loop into a self-contained entry-point function suitable for a managed container
- Log training loss as CloudWatch metrics with `log_metric` and persist progress with job checkpoints
- Submit the job with `AwsQuantumJob.create`, guarded behind `RUN_ON_AWS = False` with a cost note

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Use local simulator by default (free, instant). Everything that EXECUTES in this
# notebook runs here -- no AWS session is created and no AWS API is called.
device = LocalSimulator()

# Reproducibility: seed NumPy so the training curve below is identical every run.
np.random.seed(7)

## 1. The model: a 2-qubit variational quantum classifier

The model is a quantum circuit (the GUIDE's central idea). For a 2-feature input `x = (x0, x1)` we:

1. **Angle-encode** the data: `RY(0, x0)` and `RY(1, x1)` rotate each qubit by its feature.
2. **Entangle and rotate** with a trainable layer parameterized by `theta = (w0, w1, w2, w3)`.
3. **Read out** the prediction as the expectation `<Z_0>`, which lives in `[-1, +1]` -- a natural
   binary-classifier output (`+1` for one class, `-1` for the other).

The encoding is fixed; only `theta` is learned. We keep it deliberately tiny -- 2 qubits, four
trainable angles -- so it trains in seconds on the local simulator and is cheap to later run as a
managed job.

In [ ]:
def build_vqc(x, params):
    """2-qubit VQC: angle-encode x, then a trainable entangling layer.

    x      : length-2 feature vector (radians)
    params : length-4 trainable angles (w0, w1, w2, w3)
    """
    w0, w1, w2, w3 = params
    c = Circuit()
    c.ry(0, x[0]); c.ry(1, x[1])      # data encoding (input layer)
    c.cnot(0, 1)                       # entangle
    c.ry(0, w0); c.ry(1, w1)          # trainable rotations (hidden layer)
    c.cnot(1, 0)
    c.ry(0, w2); c.ry(1, w3)
    return c

# Inspect the circuit at sample parameters.
demo = build_vqc(x=[0.3, 0.2], params=[0.5, 1.0, 1.5, 2.0])
print(demo)
print("qubits:", demo.qubit_count, " depth:", demo.depth)

## 2. Reading the prediction: `<Z_0>` from counts

The classifier's output is the expectation of `Z` on qubit 0. We estimate it by sampling: run the
circuit for `shots`, then average `+1` for each shot where qubit 0 reads `0` and `-1` where it
reads `1`. Braket's `measurement_counts` are big-endian, so `bitstring[0]` is qubit 0 (same
convention as the QAOA notebook). This is a *statistical* estimate, so it carries shot noise -- we
use loose, noise-aware thresholds, never exact equality.

In [ ]:
def expectation_z0(x, params, device, shots):
    """Monte-Carlo estimate of <Z_0> in [-1, +1] from measurement counts."""
    counts = device.run(build_vqc(x, params), shots=shots).result().measurement_counts
    total = sum(counts.values())
    # Big-endian: bitstring[0] is qubit 0. Z eigenvalue is +1 for '0', -1 for '1'.
    signed = sum((1.0 if b[0] == "0" else -1.0) * n for b, n in counts.items())
    return signed / total

# A point encoded near |0...> should read <Z_0> close to +1 at identity-ish params.
val = expectation_z0([0.3, 0.2], [0.5, 1.0, 1.5, 2.0], device, shots=2000)
print("sample <Z_0> =", round(val, 4))
assert -1.0 <= val <= 1.0, "an expectation of Z must lie in [-1, +1]"
# Proven: the readout is a well-formed expectation value bounded by the spectrum of Z.

## 3. The training loop: loss + parameter-shift gradients

This is the loop that will *become the job*. We use a tiny labeled dataset (two points per class),
a mean-squared-error loss between `<Z_0>` and the target label, and the **parameter-shift rule** for
exact gradients: for each trainable angle, evaluate the loss at `theta + pi/2` and `theta - pi/2`
and take half their difference (the GUIDE's two-evaluation exact-gradient trick). Then take a
gradient-descent step. Everything here runs on `LocalSimulator` -- this is the algorithm we will
lift into a managed container in Section 5, written so it is already self-contained.

In [ ]:
def loss_fn(X, Y, params, device, shots):
    """Mean-squared error between predicted <Z_0> and target labels Y in {-1, +1}."""
    preds = np.array([expectation_z0(x, params, device, shots) for x in X])
    return float(np.mean((preds - Y) ** 2))

def param_shift_grad(X, Y, params, device, shots):
    """Exact gradient of loss_fn via the parameter-shift rule (2 evals per angle)."""
    grad = np.zeros_like(params)
    s = np.pi / 2
    for k in range(len(params)):
        plus = params.copy();  plus[k]  += s
        minus = params.copy(); minus[k] -= s
        grad[k] = 0.5 * (loss_fn(X, Y, plus, device, shots)
                         - loss_fn(X, Y, minus, device, shots))
    return grad

# Tiny 2-class dataset: class +1 near the origin, class -1 near (pi, pi).
X = np.array([[0.3, 0.2], [0.1, 0.4], [2.9, 2.7], [2.6, 3.0]])
Y = np.array([1.0, 1.0, -1.0, -1.0])
print("dataset X:\n", X)
print("labels  Y:", Y)

## 4. Train locally and watch the loss fall

Now run the loop. We start from random angles (seeded, so the curve is reproducible), take a few
parameter-shift gradient steps, and print the loss at each step. Because we deliberately initialize
*far* from the solution, the loss starts high (~2.4, near the worst case of 4 for MSE on labels in
`{-1,+1}`) and collapses toward 0 within a handful of steps -- the reader sees real learning with no
AWS involved.

In [ ]:
def train(X, Y, steps=8, lr=0.5, shots=2000, init=None):
    """Self-contained training loop. Returns (loss_history, final_params).

    Written so it can run unchanged inside a Braket Hybrid Job container.
    """
    params = np.random.uniform(0, 2 * np.pi, size=4) if init is None else np.array(init)
    history = []
    for t in range(steps):
        L = loss_fn(X, Y, params, device, shots)
        history.append(L)
        print(f"step {t:2d}  loss = {L:.4f}")
        params = params - lr * param_shift_grad(X, Y, params, device, shots)
    final_loss = loss_fn(X, Y, params, device, shots)
    history.append(final_loss)
    print(f"final     loss = {final_loss:.4f}")
    return history, params

history, trained_params = train(X, Y, steps=8, lr=0.5, shots=2000)

# Headline claim: the loss genuinely decreased. Loose check (shot noise): final < half of start.
assert history[-1] < history[0] * 0.5, "training should at least halve the loss"
assert history[-1] < 0.05, "a separable toy problem should reach near-zero loss"
# Proven: a real VQC, trained on LocalSimulator with parameter-shift gradients, learns the task.

## 5. Plot the learning curve

The loss history *is* the signal we will ship to CloudWatch as a metric once this runs as a job. Here
we just plot it locally to confirm the descent.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(range(len(history)), history, marker="o", color="#5b6ef5")
ax.set_xlabel("training step")
ax.set_ylabel("MSE loss")
ax.set_title("Local VQC training on LocalSimulator (seed=7)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("loss[0] =", round(history[0], 4), "  loss[-1] =", round(history[-1], 4))

## 6. The job entry point: log metrics and checkpoint

A Braket Hybrid Job runs a Python **entry point** inside a managed container, on a device you choose
(here the managed SV1 simulator). Inside that container the SDK gives you three production hooks:

- **`log_metric`** (`braket.jobs.metrics`) emits a named scalar that Braket forwards to CloudWatch,
  so you can watch the loss curve live in the console while the job runs.
- **`save_job_checkpoint` / `load_job_checkpoint`** (`braket.jobs`) persist and resume optimizer
  state, so a long run can recover after an interruption instead of restarting from scratch.
- **`save_job_result`** (`braket.jobs`) writes the final artifact (the trained parameters) to the
  job's S3 output.

The entry point also reads the device ARN from the environment (`get_job_device_arn`) so the same
code runs on the local simulator during development and on SV1 (or a QPU) in production. The cell
below shows the entry point as a **string** -- it is the file you would hand to the job. It is *not*
executed here (it calls job-only APIs that require the managed runtime), it is shown for reading.

In [ ]:
# This is the entry-point SOURCE you would save as `vqc_job.py`. It is printed, not run:
# the log_metric / checkpoint / save_job_result calls only work inside the managed
# Braket Hybrid Job container, never at notebook top level.
entry_point_source = '''
import numpy as np
from braket.aws import AwsDevice
from braket.circuits import Circuit
from braket.devices import LocalSimulator
from braket.jobs import save_job_checkpoint, load_job_checkpoint, save_job_result
from braket.jobs.metrics import log_metric
from braket.jobs.environment_variables import get_job_device_arn


def build_vqc(x, params):
    w0, w1, w2, w3 = params
    c = Circuit()
    c.ry(0, x[0]); c.ry(1, x[1]); c.cnot(0, 1)
    c.ry(0, w0); c.ry(1, w1); c.cnot(1, 0); c.ry(0, w2); c.ry(1, w3)
    return c


def expectation_z0(x, params, device, shots):
    counts = device.run(build_vqc(x, params), shots=shots).result().measurement_counts
    total = sum(counts.values())
    return sum((1.0 if b[0] == "0" else -1.0) * n for b, n in counts.items()) / total


def main():
    # Hyperparameters arrive as a dict of strings; cast them.
    import json, os
    hp = json.loads(os.environ.get("AMZN_BRAKET_HP_FILE_PATH_CONTENTS", "{}")) \\
        if "AMZN_BRAKET_HP_FILE_PATH_CONTENTS" in os.environ else {}
    steps = int(hp.get("steps", 30))
    lr = float(hp.get("lr", 0.5))
    shots = int(hp.get("shots", 2000))

    # One-line device switch: managed SV1 (or QPU) in the job, local in dev.
    arn = get_job_device_arn()
    device = AwsDevice(arn) if arn else LocalSimulator()

    X = np.array([[0.3, 0.2], [0.1, 0.4], [2.9, 2.7], [2.6, 3.0]])
    Y = np.array([1.0, 1.0, -1.0, -1.0])

    # Resume from a checkpoint if one exists, else start fresh.
    try:
        ckpt = load_job_checkpoint(checkpoint_file_suffix="latest")
        params = np.array(ckpt["params"]); start = int(ckpt["step"])
    except Exception:
        np.random.seed(7); params = np.random.uniform(0, 2 * np.pi, size=4); start = 0

    for t in range(start, steps):
        preds = np.array([expectation_z0(x, params, device, shots) for x in X])
        loss = float(np.mean((preds - Y) ** 2))
        log_metric(metric_name="loss", value=loss, iteration_number=t)  # -> CloudWatch
        # parameter-shift gradient
        grad = np.zeros_like(params); s = np.pi / 2
        for k in range(len(params)):
            pp = params.copy(); pp[k] += s; pm = params.copy(); pm[k] -= s
            lp = float(np.mean((np.array([expectation_z0(x, pp, device, shots) for x in X]) - Y) ** 2))
            lm = float(np.mean((np.array([expectation_z0(x, pm, device, shots) for x in X]) - Y) ** 2))
            grad[k] = 0.5 * (lp - lm)
        params = params - lr * grad
        save_job_checkpoint({"params": params.tolist(), "step": t + 1},
                            checkpoint_file_suffix="latest")

    save_job_result({"trained_params": params.tolist()})


if __name__ == "__main__":
    main()
'''

print(entry_point_source[:600], "...\n[truncated; full source in the string above]")
print("\nentry-point length (chars):", len(entry_point_source))

## 7. Confirm the job APIs import (no AWS call)

Before submitting anything, sanity-check that the Hybrid Jobs symbols are importable. Importing them
only loads classes -- it does **not** create an AWS session or contact AWS. Note `AwsQuantumJob` lives
in `braket.aws` (not `braket.jobs`).

In [ ]:
# These imports succeed offline (boto3 installed) and create NO session / make NO API call.
try:
    from braket.aws import AwsQuantumJob          # the job submitter
    from braket.jobs.metrics import log_metric    # CloudWatch metric emitter
    from braket.jobs import (                      # checkpoint + result helpers
        save_job_checkpoint, load_job_checkpoint, save_job_result,
    )
    print("Hybrid Jobs APIs available:")
    print("  AwsQuantumJob       <-", AwsQuantumJob.__module__)
    print("  log_metric          <-", log_metric.__module__)
    print("  save_job_checkpoint <-", save_job_checkpoint.__module__)
except ImportError as e:
    print("Hybrid Jobs SDK not installed in this environment:", e)
    print("Install with: pip install amazon-braket-sdk  (then `make setup`)")
# Proven: the packaging APIs resolve without any AWS credentials or network call.

## 8. Submit the job -- guarded behind `RUN_ON_AWS = False`

Finally, the submission. `AwsQuantumJob.create` uploads the entry point, spins up a managed
container, and runs the training on the device ARN you pass -- here the **managed SV1 simulator**.
This is the only step that incurs cost and requires credentials, so it is **guarded**: `RUN_ON_AWS`
defaults to `False`, and the `create` call lives inside `if RUN_ON_AWS:`, so running this notebook
top-to-bottom never submits anything.

To actually run it, save `entry_point_source` to `vqc_job.py`, set `RUN_ON_AWS = True`, and provide a
job-execution `role_arn`. **Cost note:** SV1 bills about $0.075-$0.275 per minute of container time;
a short job like this runs a few minutes (single-digit dollars). It uses the managed *simulator* by
default -- it touches a real QPU only if you deliberately change `device_arn` to a hardware ARN,
which adds per-shot + per-task charges. Always validate on `LocalSimulator` first (Sections 1-5).

In [ ]:
RUN_ON_AWS = False  # <-- default OFF. Flip to True only when you intend to spend money.

# Managed SV1 simulator ARN (NOT a QPU). Cost: ~$0.075-$0.275 / minute of job runtime.
SV1_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"

if RUN_ON_AWS:
    # Cost-incurring path. Requires AWS credentials + a Braket job-execution role.
    # Estimated spend: a few minutes of SV1 container time (single-digit USD).
    from pathlib import Path
    from braket.aws import AwsQuantumJob

    Path("vqc_job.py").write_text(entry_point_source)

    job = AwsQuantumJob.create(
        device=SV1_ARN,                       # managed simulator, not hardware
        source_module="vqc_job.py",
        entry_point="vqc_job:main",
        job_name="vqc-hybrid-demo",
        hyperparameters={"steps": 30, "lr": 0.5, "shots": 2000},
        # role_arn="arn:aws:iam::<account-id>:role/AmazonBraketJobsExecutionRole",
        wait_until_complete=False,            # poll/stream instead of blocking
    )
    print("submitted:", job.arn)
    print("loss metric will stream to CloudWatch; fetch with job.metrics() when running")
else:
    print("RUN_ON_AWS is False -- no job submitted, no cost incurred.")
    print("Validated locally above; set RUN_ON_AWS = True (and a role_arn) to run on SV1.")

## Exercises

The local simulator is free -- experiment freely. The AWS-gated cells are for reading and adapting;
leave `RUN_ON_AWS = False` unless you intend to spend.

In [ ]:
# Exercise 1: Longer local training + a learning-rate sweep.
# Re-run train() with steps=20 for lr in {0.1, 0.5, 1.0}. Plot all three loss curves on
# one axis. Which lr converges fastest? Does the largest lr overshoot (loss bounces)?
#
# for lr in (0.1, 0.5, 1.0):
#     np.random.seed(7)  # same init each time for a fair comparison
#     hist, _ = train(X, Y, steps=20, lr=lr, shots=2000)
#     plt.plot(hist, label=f"lr={lr}")
# plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

In [ ]:
# Exercise 2: Add a checkpoint/resume to the LOCAL loop (mirrors the job's behavior).
# Modify train() to (a) accept a `resume_from` dict {'params':..., 'step':...},
# (b) return that dict each step, and (c) start from it if provided. Run 4 steps, capture
# the state, then run 4 more starting from it -- the combined loss curve should match a
# single 8-step run with the same seed. This is exactly what save_job_checkpoint enables
# in the managed container, minus the S3 round-trip.
#
# def train_resumable(X, Y, steps, lr, shots, resume_from=None): ...
# state = train_resumable(X, Y, steps=4, lr=0.5, shots=2000)
# state = train_resumable(X, Y, steps=4, lr=0.5, shots=2000, resume_from=state)

## Summary

- A **QML model is a circuit**: we built a 2-qubit VQC that angle-encodes 2 features, applies a
  trainable entangling layer, and reads out the prediction as `<Z_0>` in `[-1, +1]`.
- The **training loop runs on `LocalSimulator`**: parameter-shift gradients (two evaluations per
  angle, exact) plus gradient descent drove the MSE loss from ~2.4 to near 0 in 8 steps, seeded for
  reproducibility -- real learning with no AWS.
- That same loop becomes a **job entry point**: inside a managed container it would call
  `log_metric` (loss -> CloudWatch), `save_job_checkpoint` / `load_job_checkpoint` (resume long
  runs), and `save_job_result` (persist trained params), switching device via `get_job_device_arn`.
- **Submission is `AwsQuantumJob.create`** (imported from `braket.aws`), pointed at the managed SV1
  simulator with hyperparameters -- guarded behind `RUN_ON_AWS = False` so the notebook never spends
  money. SV1 bills ~$0.075-$0.275/min; a QPU is used only if you deliberately swap the ARN.
- The workflow is the production discipline from `CLAUDE.md`: prototype on the free local simulator,
  validate, *then* gate the managed run behind an explicit cost-aware flag.

**Next:** [`../../05-quantum-chemistry/notebooks/01-molecular-hamiltonians.ipynb`](../../05-quantum-chemistry/notebooks/01-molecular-hamiltonians.ipynb)
-- point the same variational machinery at molecules with the Variational Quantum Eigensolver (VQE).